# 01 — Foundations: Tensors, Shapes, Devices, and Autograd

Goal: become fluent with tensor semantics and PyTorch fundamentals.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

In [ ]:

def seed_all(seed=1234):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(42)

## 1. Tensor creation, dtype, device

Core properties:
- `shape`, `dtype`, `device`
- `requires_grad`
- contiguous vs non-contiguous layouts

In [ ]:

import torch

a = torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32, device=device)
b = torch.zeros((2,3), device=device)
c = torch.randn((2,3), device=device)
d = torch.arange(0, 12, device=device).reshape(3,4)

print(a)
print("dtype:", a.dtype, "device:", a.device, "shape:", a.shape)
print("contiguous:", d.is_contiguous())

### Common factories
- `zeros/ones/empty`
- `rand/randn/randint`
- `linspace/logspace`
- `eye/diag`
- numpy interoperability (`from_numpy`, `.numpy()` CPU-only)

In [ ]:

x = torch.linspace(0, 1, 5, device=device)
y = torch.randint(low=0, high=10, size=(3,4), device=device)
I = torch.eye(4, device=device)
print(x)
print("y shape:", y.shape)
print("I shape:", I.shape)

## 2. Strides, views, reshape, permute

PyTorch tensors are strided views over storage.
Key APIs:
- `.stride()`
- `.view()` requires contiguous
- `.reshape()` may copy if needed
- `.transpose()` / `.permute()` usually produce non-contiguous views

In [ ]:

t = torch.arange(12, device=device).reshape(3,4)
tT = t.transpose(0,1)
print("t:", t.shape, t.stride(), "contig:", t.is_contiguous())
print("tT:", tT.shape, tT.stride(), "contig:", tT.is_contiguous())

try:
    tT.view(-1)
except RuntimeError as e:
    print("view failed (expected):", str(e).splitlines()[0])

flat = tT.reshape(-1)
print("reshape ok:", flat.shape)

## 3. Broadcasting and reduction

Broadcast aligns dims from the right; dims must match or be 1.
Reductions: `sum/mean/max/min`, `keepdim=True` to preserve dimensions.

In [ ]:

A = torch.randn(10, 1, device=device)
B = torch.randn(1, 20, device=device)
C = A + B
print("broadcast shape:", C.shape)

u = torch.randn(3,4, device=device)
print("sum over dim=1:", u.sum(dim=1).shape)
print("keepdim:", u.sum(dim=1, keepdim=True).shape)

## 4. Indexing, masks, gather/scatter

- boolean masks for filtering
- integer indexing tensors for advanced selection
- `gather` pulls values from indices
- `scatter_` writes values at indices

In [ ]:

x = torch.arange(0, 24, device=device).reshape(2,3,4)
print("x[0]:\n", x[0])
print("x[:,:,1]:\n", x[:,:,1])

mask = x > 10
print("num>10:", int(mask.sum()))

src = torch.tensor([[10,11,12],[20,21,22]], device=device)
ind = torch.tensor([[2,0],[1,1]], device=device)
g = torch.gather(src, dim=1, index=ind)
print("gather:\n", g)

out = torch.zeros_like(src)
out.scatter_(dim=1, index=ind, src=torch.tensor([[9,9],[7,7]], device=device))
print("scatter:\n", out)

## 5. Autograd core

- set `requires_grad=True`
- `loss.backward()` computes gradients
- grads accumulate; clear them between steps
- use `torch.no_grad()` / `torch.inference_mode()` for inference

In [ ]:

x = torch.tensor(2.0, device=device, requires_grad=True)
y = 3*x**2 + 2*x + 1
y.backward()
print("dy/dx:", x.grad)  # 14

## 6. Numerical stability basics

- use logits-based losses (`CrossEntropyLoss`, `BCEWithLogitsLoss`)
- use `logsumexp` instead of `log(sum(exp(x)))`

In [ ]:

v = torch.randn(4, device=device)
stable = torch.logsumexp(v, dim=0)
naive = torch.log(torch.exp(v).sum())
print("abs diff:", float((stable - naive).abs()))

## 7. Seeding and generators

Determinism can be difficult across devices/backends; seed for debugging.

In [ ]:

import random, numpy as np, torch
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print(torch.rand(3, device=device))